In [11]:
import pandas as pd
import duckdb
import time
import os

# Set file path to the user's home directory for 'paysim dataset.csv'
user_home = os.path.expanduser('~')
csv_path = os.path.join(user_home, 'paysim dataset.csv')

In [12]:
# 1. Start timer
start_time = time.time()

# 2. SQL Query
sql_query = f"""
SELECT 
    nameOrig AS customer_id,
    COUNT(*) AS total_txn_count,
    ROUND(SUM(amount), 2) AS total_amount,
    ROUND(AVG(amount), 2) AS avg_txn_amount,
    ROUND(MAX(amount), 2) AS max_single_txn
FROM '{csv_path}'
WHERE type = 'CASH_IN'
GROUP BY nameOrig
ORDER BY total_amount DESC;
"""

# 3. Execute query and convert to Pandas DataFrame
df_sql_alerts = duckdb.query(sql_query).to_df()

# 4. Performance & Output Display
print(f"SQL Execution Time: {time.time() - start_time:.2f} seconds")
print(f"Total High-Risk CASH_IN Customers Identified: {len(df_sql_alerts)}")
df_sql_alerts.head(10)

SQL Execution Time: 0.70 seconds
Total High-Risk CASH_IN Customers Identified: 1398845


,customer_id,total_txn_count,total_amount,avg_txn_amount,max_single_txn
0,C1603141931,1,1915267.90,1915267.90,1915267.90
1,C1072983741,1,1821279.53,1821279.53,1821279.53
2,C1719006755,1,1782621.49,1782621.49,1782621.49
3,C866302906,1,1781905.26,1781905.26,1781905.26
4,C1139859398,1,1675137.60,1675137.60,1675137.60
5,C1931735461,1,1609288.13,1609288.13,1609288.13
6,C1065575100,1,1602113.20,1602113.20,1602113.20
7,C1130769687,1,1588281.27,1588281.27,1588281.27
8,C1747502623,1,1573923.99,1573923.99,1573923.99
9,C263860433,1,1529738.97,1529738.97,1529738.97


In [13]:
# 1. Start timer
start_time = time.time()

# 2. Load only essential columns (RAM-optimized approach)
use_cols = ['type', 'amount', 'nameOrig']
df = pd.read_csv(csv_path, usecols=use_cols)

# 3. Filtering (Equivalent to: WHERE type = 'CASH_IN')
df_filtered = df[df['type'] == 'CASH_IN'].copy()

# 4. Grouping & Aggregation (Equivalent to: GROUP BY nameOrig)
df_pandas_alerts = df_filtered.groupby('nameOrig').agg(
    total_txn_count=('amount', 'count'),
    total_amount=('amount', 'sum'),
    avg_txn_amount=('amount', 'mean'),
    max_single_txn=('amount', 'max')
).reset_index()

# 5. Standardize column names to match SQL output
df_pandas_alerts = df_pandas_alerts.rename(columns={'nameOrig': 'customer_id'})

# 6. Rounding & Sorting (Equivalent to: ORDER BY total_amount DESC)
df_pandas_alerts['total_amount'] = df_pandas_alerts['total_amount'].round(2)
df_pandas_alerts['avg_txn_amount'] = df_pandas_alerts['avg_txn_amount'].round(2)
df_pandas_alerts['max_single_txn'] = df_pandas_alerts['max_single_txn'].round(2)

df_pandas_alerts = df_pandas_alerts.sort_values(by='total_amount', ascending=False)

# 7. Performance & Output Display
print(f"Pandas Execution Time: {time.time() - start_time:.2f} seconds")
print(f"Total High-Risk CASH_IN Customers Identified: {len(df_pandas_alerts)}")
df_pandas_alerts.head(10)

Pandas Execution Time: 5.37 seconds
Total High-Risk CASH_IN Customers Identified: 1398845


,customer_id,total_txn_count,total_amount,avg_txn_amount,max_single_txn
436706,C1603141931,1,1915267.90,1915267.90,1915267.90
52710,C1072983741,1,1821279.53,1821279.53,1821279.53
520635,C1719006755,1,1782621.49,1782621.49,1782621.49
1302553,C866302906,1,1781905.26,1781905.26,1781905.26
101030,C1139859398,1,1675137.60,1675137.60,1675137.60
673976,C1931735461,1,1609288.13,1609288.13,1609288.13
47256,C1065575100,1,1602113.20,1602113.20,1602113.20
94458,C1130769687,1,1588281.27,1588281.27,1588281.27
541146,C1747502623,1,1573923.99,1573923.99,1573923.99
866058,C263860433,1,1529738.97,1529738.97,1529738.97
